# 03 Matching Comparison — Соусы

Цель: сравнить research-baselines A/B на вручную размеченном `labeling_sauces.csv` без падений, пока разметка ещё пустая или отсутствует.

- A: rule-based fuzzy + brand/weight/pack fusion.
- B: zero-shot multilingual bi-encoder, если доступен `sentence-transformers`.
- Метрики считаются только по 3 классам: `exact_duplicate`, `same_product_different_pack`, `different_product`. `uncertain` остаётся для разметки, но не входит в pairwise classification report.

## План

1. Найти `research/dedup/data/labeling_sauces.csv`.
2. Если файла нет или 3-class labels ещё не заполнены — показать статус-заглушку и создать пустые таблицы.
3. Если labels есть — прогнать baseline A и baseline B.
4. Посчитать per-class precision/recall/F1, confusion matrix и false-merge ошибки.
5. Собрать сравнительную таблицу методов для следующего отчётного ноутбука.

In [ ]:
from pathlib import Path
import math
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    BiEncoderMatcher,
    RuleBasedMatcher,
    classification_report_df,
    confusion_matrix_df,
    decide_label,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 160)

In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"

EVAL_LABELS = ["exact_duplicate", "same_product_different_pack", "different_product"]
MERGE_LIKE_LABELS = {"exact_duplicate", "same_product_different_pack"}

RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "intfloat/multilingual-e5-base")

print(f"Labeling path: {LABELING_PATH}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model: {BI_ENCODER_MODEL}")

In [ ]:
def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_labeling_file",
                "message": f"Файл {path} пока не найден. Выполните 02_labeling_dataset.ipynb и заполните label.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_label_column",
                "message": "В файле нет колонки label. Перегенерируйте labeling dataset из notebook-2.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
            }
        ])
        return pd.DataFrame(), status

    labels = frame["label"].fillna("").astype(str).str.strip()
    labeled = frame[labels.isin(EVAL_LABELS)].copy()
    labeled["label"] = labels[labels.isin(EVAL_LABELS)].to_numpy()
    ignored_count = int(labels.ne("").sum() - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Gold-set готов для метрик."
        if not labeled.empty
        else "3-class labels пока не заполнены: метрики ниже будут заглушками, notebook не падает."
    )
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_non_3class": ignored_count,
        }
    ])
    return labeled.reset_index(drop=True), status


labeled_pairs, labeling_status = load_labeled_pairs(LABELING_PATH)
display(labeling_status)
if not labeled_pairs.empty:
    display(labeled_pairs["label"].value_counts().rename_axis("label").reset_index(name="pairs"))
    display(labeled_pairs.head(5))

In [ ]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)

In [ ]:
def _predict_from_score(matcher, row: pd.Series, score: float) -> str:
    if math.isnan(score):
        return "different_product"
    label = decide_label(row, score, matcher.config.fusion)
    if label == "uncertain":
        return matcher.config.uncertain_fallback_label
    return label


def run_matcher(matcher, pairs: pd.DataFrame) -> tuple[pd.DataFrame | None, str]:
    if pairs.empty:
        return None, "skipped_empty_gold_set"

    scored = pairs.copy()
    row_objects = [row for _, row in scored.iterrows()]
    if isinstance(matcher, BiEncoderMatcher):
        scores = matcher.score_batch(row_objects)
        if all(math.isnan(score) for score in scores):
            return None, matcher.status().message
        predictions = [_predict_from_score(matcher, row, score) for row, score in zip(row_objects, scores, strict=False)]
    else:
        scores = [matcher.score(row) for row in row_objects]
        predictions = [matcher.predict_label(row) for row in row_objects]

    scored[f"{matcher.name}_score"] = scores
    scored["predicted_label"] = predictions
    scored["method"] = matcher.name
    scored["false_merge"] = scored["predicted_label"].isin(MERGE_LIKE_LABELS) & scored["label"].eq("different_product")
    return scored, "ready"

In [ ]:
method_outputs: dict[str, pd.DataFrame] = {}
method_reports: dict[str, pd.DataFrame] = {}
method_confusions: dict[str, pd.DataFrame] = {}
false_merge_examples: dict[str, pd.DataFrame] = {}
skipped_methods: list[dict[str, str]] = []
summary_rows: list[dict[str, object]] = []

if labeled_pairs.empty:
    placeholder_report = pd.DataFrame(
        [{"method": "not_available_yet", "label": label, "precision": None, "recall": None, "f1": None, "support": 0} for label in EVAL_LABELS]
    )
    placeholder_summary = pd.DataFrame(
        [
            {
                "method": "not_available_yet",
                "status": labeling_status.loc[0, "status"],
                "pairs": 0,
                "macro_precision": None,
                "macro_recall": None,
                "macro_f1": None,
                "false_merge_count": None,
                "false_merge_rate": None,
            }
        ]
    )
    display(placeholder_summary)
    display(placeholder_report)
else:
    for matcher in matchers:
        result, status = run_matcher(matcher, labeled_pairs)
        if result is None:
            skipped_methods.append({"method": matcher.name, "status": status})
            continue

        report = classification_report_df(result["label"], result["predicted_label"], labels=EVAL_LABELS)
        report.insert(0, "method", matcher.name)
        confusion = confusion_matrix_df(result["label"], result["predicted_label"], labels=EVAL_LABELS)
        false_merges = result[result["false_merge"]].copy()

        method_outputs[matcher.name] = result
        method_reports[matcher.name] = report
        method_confusions[matcher.name] = confusion
        false_merge_examples[matcher.name] = false_merges
        summary_rows.append(
            {
                "method": matcher.name,
                "status": "ready",
                "pairs": len(result),
                "macro_precision": report["precision"].mean(),
                "macro_recall": report["recall"].mean(),
                "macro_f1": report["f1"].mean(),
                "exact_duplicate_precision": report.loc[report["label"].eq("exact_duplicate"), "precision"].iloc[0],
                "false_merge_count": int(false_merges.shape[0]),
                "false_merge_rate": float(false_merges.shape[0] / len(result)) if len(result) else 0.0,
            }
        )

    for skipped in skipped_methods:
        summary_rows.append(
            {
                "method": skipped["method"],
                "status": skipped["status"],
                "pairs": 0,
                "macro_precision": None,
                "macro_recall": None,
                "macro_f1": None,
                "exact_duplicate_precision": None,
                "false_merge_count": None,
                "false_merge_rate": None,
            }
        )

    comparison_table = pd.DataFrame(summary_rows)
    display(comparison_table)
    if method_reports:
        display(pd.concat(method_reports.values(), ignore_index=True))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))

In [ ]:
if not method_confusions:
    print("Confusion matrices появятся после заполнения 3-class labels.")
else:
    for method, confusion in method_confusions.items():
        print(f"Confusion matrix: {method}")
        display(confusion)

if not false_merge_examples:
    print("False-merge examples появятся после запуска хотя бы одного метода на размеченных парах.")
else:
    for method, examples in false_merge_examples.items():
        print(f"False merge examples: {method} ({len(examples)})")
        columns = [
            "label",
            "predicted_label",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "unit_amount_a",
            "unit_amount_b",
            "total_amount_a",
            "total_amount_b",
            "multipack_count_a",
            "multipack_count_b",
        ]
        display(examples[[column for column in columns if column in examples.columns]].head(20))

## Следующие шаги

- После ручной разметки перезапустить notebook сверху вниз.
- На dev-set откалибровать `threshold_high` и `threshold_low` отдельно для A/B.
- Для отчёта вынести false-merge примеры в качественный анализ: это самая дорогая ошибка из раздела 8.3.